In [1]:
import os
import pandas as pd
import numpy as np

PROCESSED_DIR = os.path.join("..", "data", "processed")
REPORTS_DIR = os.path.join("..", "reports")

In [4]:
PORTFOLIO_VALUE = 100000.0  # Total portfolio target in USD

# Load recommendations generated from notebook 06
rec_path = os.path.join(REPORTS_DIR, "recommendations.csv")
df = pd.read_csv(rec_path)

# Ensure fallback values if specific columns aren't present
if "Weight" not in df.columns:
    df["Weight"] = 1.0 / len(df)  # Equal weighting fallback

if "Recommendation Signal" not in df.columns and "Signal" in df.columns:
    df["Recommendation Signal"] = df["Signal"]
elif "Recommendation Signal" not in df.columns:
    df["Recommendation Signal"] = "BUY"

In [5]:

# Mock current market prices if not present
if "Current Price ($)" not in df.columns:
    # Example mapping or dummy price calculation
    default_prices = {"AAPL": 180.50, "MSFT": 420.10, "GOOGL": 175.25, "AMZN": 185.00}
    df["Current Price ($)"] = df["Ticker"].map(default_prices).fillna(150.0)

# Calculate target allocation and share quantities
df["Target Allocation ($)"] = df["Weight"] * PORTFOLIO_VALUE
df["Target Shares"] = (df["Target Allocation ($)"] / df["Current Price ($)"]).astype(int)

# Determine order type based on recommendation signal
df["Order Type"] = df["Recommendation Signal"].apply(
    lambda x: "BUY" if "BUY" in str(x).upper() else ("SELL" if "SELL" in str(x).upper() else "HOLD")
)

df[["Ticker", "Current Price ($)", "Weight", "Target Allocation ($)", "Target Shares", "Order Type"]]

,Ticker,Current Price ($),Weight,Target Allocation ($),Target Shares,Order Type
0,AAPL,180.5,0.083333,8333.333333,46,BUY
1,CAT,150.0,0.083333,8333.333333,55,HOLD
2,HD,150.0,0.083333,8333.333333,55,HOLD
3,JNJ,150.0,0.083333,8333.333333,55,HOLD
4,JPM,150.0,0.083333,8333.333333,55,HOLD
5,KO,150.0,0.083333,8333.333333,55,HOLD
6,MSFT,420.1,0.083333,8333.333333,19,HOLD
7,NVDA,150.0,0.083333,8333.333333,55,HOLD
8,PG,150.0,0.083333,8333.333333,55,HOLD
9,XOM,150.0,0.083333,8333.333333,55,HOLD


In [6]:
rebalancing_orders = df[[
    "Ticker", "Current Price ($)", "Weight", 
    "Target Allocation ($)", "Target Shares", "Order Type"
]]

rebalancing_orders.columns = [
    "Ticker", "Price ($)", "Optimal Weight", 
    "Target Value ($)", "Shares to Trade", "Action"
]

rebalancing_orders

,Ticker,Price ($),Optimal Weight,Target Value ($),Shares to Trade,Action
0,AAPL,180.5,0.083333,8333.333333,46,BUY
1,CAT,150.0,0.083333,8333.333333,55,HOLD
2,HD,150.0,0.083333,8333.333333,55,HOLD
3,JNJ,150.0,0.083333,8333.333333,55,HOLD
4,JPM,150.0,0.083333,8333.333333,55,HOLD
5,KO,150.0,0.083333,8333.333333,55,HOLD
6,MSFT,420.1,0.083333,8333.333333,19,HOLD
7,NVDA,150.0,0.083333,8333.333333,55,HOLD
8,PG,150.0,0.083333,8333.333333,55,HOLD
9,XOM,150.0,0.083333,8333.333333,55,HOLD


In [7]:
output_path = os.path.join(REPORTS_DIR, "rebalancing_orders.csv")
rebalancing_orders.to_csv(output_path, index=False)
print(f"Successfully generated rebalancing orders and saved to {output_path}")

Successfully generated rebalancing orders and saved to ..\reports\rebalancing_orders.csv
